# Stage 02: Tooling setup check

This notebook verifies the Python environment, local configuration, project paths, and a small SPY risk calculation.

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install python-dotenv

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/paritoshdwivedi/Downloads/project bootcamp/bootcamp_paritosh_dwivedi/homework/homework02

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


## 1. Interpreter and version
Report the runtime so the submitted outputs document the environment used for execution.

In [3]:
# Report both the Python build and the executable selected by Jupyter.
import sys

print("Python version:", sys.version)
print("Interpreter path:", sys.executable)

Python version: 3.14.3 (main, Feb  3 2026, 15:32:20) [Clang 17.0.0 (clang-1700.6.3.2)]
Interpreter path: /opt/homebrew/opt/python@3.14/bin/python3.14


## 2. Load the environment file
`python-dotenv` reads the dummy settings from `.env` without placing configuration values in notebook code.

In [4]:
# The root-level submission notebook starts in the homework folder.
from dotenv import load_dotenv

environment_loaded = load_dotenv(dotenv_path=Path(".env"), override=True)
print("Loaded .env:", environment_loaded)

Loaded .env: True


## 3. Reusable configuration helpers
The notebook imports `load_env` and `get_key` from `src/config.py`. Importing one implementation avoids drift between notebooks and gives the later Weekly ETF Risk Monitor a reusable configuration boundary.

In [5]:
from src.config import get_key, load_env

environment_loaded_by_helper = load_env()
homework_root = Path.cwd()
data_dir = Path(get_key("DATA_DIR", "./data"))

print("Config helper loaded .env:", environment_loaded_by_helper)
print("Homework folder:", homework_root.name)
print("Configured data folder:", data_dir.as_posix())

Config helper loaded .env: True
Homework folder: homework02
Configured data folder: data


## 4. Verify keys and paths
Confirm that the dummy key was loaded and that every required Stage 02 folder is present.

In [6]:
api_key_present = bool(get_key("API_KEY"))
expected_folders = (
    Path("data/raw"),
    Path("data/processed"),
    Path("notebooks"),
    Path("src"),
    Path("docs"),
    Path("reports"),
    Path("model"),
)
folders_present = data_dir.is_dir() and all(path.is_dir() for path in expected_folders)

print("API_KEY present:", api_key_present)
print("DATA_DIR present:", data_dir.is_dir())
print("All seven folders present:", folders_present)

if not api_key_present or not folders_present:
    raise RuntimeError("Required configuration or folders are missing.")

API_KEY present: True
DATA_DIR present: True
All seven folders present: True


## 5. NumPy mini-demo
The array represents five example SPY daily returns. Squaring every observation in one vectorized expression mirrors a building block of realized-volatility calculations.

In [7]:
import numpy as np

spy_daily_returns = np.array([0.004, -0.007, 0.011, 0.002, -0.003])
squared_returns = np.square(spy_daily_returns)
annualized_realized_volatility = np.sqrt(squared_returns.mean() * 252)

print("SPY daily returns:", spy_daily_returns)
print("Vectorized squared returns:", squared_returns)
print(f"Five-session annualized realized volatility: {annualized_realized_volatility:.2%}")

SPY daily returns: [ 0.004 -0.007  0.011  0.002 -0.003]
Vectorized squared returns: [1.60e-05 4.90e-05 1.21e-04 4.00e-06 9.00e-06]
Five-session annualized realized volatility: 10.01%


## Notes
- Replace dummy API keys with real ones only in your private `.env` (never commit secrets).
- Ensure this notebook runs top-to-bottom without errors.